In [ ]:
import numpy as np
import gds_geometry_evaluator as gds

# Resonator Designs

In [ ]:
N_resonators = 5

chip_h_um = 10.0e3
chip_w_um = 22.0e3

gs_trace_width = 80
gs_gap_width = 20

cpw_gap_width = 10.75
cpw_gnd_width = 100

built_lekid_list = [None] * N_resonators

## Low-Volume Design 1

In [ ]:
lekid_idx = 0
built_lekid_list[lekid_idx] = gds.build_double_meander_with_idc(
                            inductor_spec=gds.DoubleMeanderSpec(
                                trace_width_microns=4.0,
                                gap_width_microns=4.0,
                                bounding_box_width_microns=284.0,
                                bounding_box_height_microns=250.0,
                                meander_inner_gap_width_microns=4.0,
                            ),
                            idc_spec=gds.IdcSpec(
                                finger_count=16,
                                finger_trace_width_microns=4.0,
                                finger_length_microns=260.0,
                                finger_gap_width_microns=8.0,
                                arm_trace_width_microns=6.0,
                                arm_gap_width_microns=12.0,
                                bounding_box_width_microns=284.0,
                                final_finger_length_fraction=1.0,
                                include_bottom_bars=True,
                                omit_top_arm_stubs=False,
                            ),
                            ground_shield_spec=gds.GroundShieldSpec(
                                trace_thickness_microns=gs_trace_width,
                                horizontal_gap_microns=gs_gap_width,
                                upper_gap_microns=gs_gap_width,
                                lower_gap_microns=gs_gap_width,
                            ),
                        )

built_lekid_list[lekid_idx].plot()

In [ ]:
lekid_eval = gds.evaluate_generated_double_meander_with_idc(built_lekid_list[lekid_idx], 
                                               effective_permittivity=9.08, 
                                               kinetic_inductance_per_square_nh=0.0, 
                                               film_thickness_nanometers=30.0
                                              )

## MLA5G Narrow Trace

In [ ]:
lekid_idx = 1
built_lekid_list[lekid_idx] = gds.build_double_meander_with_idc(
                            inductor_spec=gds.DoubleMeanderSpec(
                                trace_width_microns=8.0,
                                gap_width_microns=2.0,
                                meander_inner_gap_width_microns=2.0,
                                bounding_box_width_microns=1130.0,
                                bounding_box_height_microns=404.0,
                                flux_trap_spec=gds.FluxTrapSpec(
                                    trap_width_microns=4.0,
                                    horizontal_separation_microns=6.0,
                                    edge_margin_microns=2.0,
                                ),
                            ),
                            idc_spec=gds.IdcSpec(
                                finger_count=18,
                                finger_trace_width_microns=20.0,
                                finger_length_microns=950.0,
                                finger_gap_width_microns=20.0,
                                arm_trace_width_microns=80.0,
                                arm_gap_width_microns=10.0,
                                bounding_box_width_microns=1130.0,
                                final_finger_length_fraction=1.0,
                                include_bottom_bars=True,
                                omit_top_arm_stubs=True,
                            ),
                            ground_shield_spec=gds.GroundShieldSpec(
                                trace_thickness_microns=gs_trace_width,
                                horizontal_gap_microns=gs_gap_width,
                                upper_gap_microns=gs_gap_width,
                                lower_gap_microns=gs_gap_width,
                            ),
                            idc_flux_trap_spec=gds.FluxTrapSpec(
                                trap_width_microns=5.0,
                                horizontal_separation_microns=6.0,
                                edge_margin_microns=4.0,
                            ),
                        )

built_lekid_list[lekid_idx].plot()

In [ ]:
fl_launch_spec = gds.FeedlineLauncherSpec(
    central_conductor_width_microns=20,
    gap_width_microns=cpw_gap_width,
    ground_conductor_width_microns=cpw_gnd_width,
    signal_bond_pad_edge_width_microns=300,
    ground_bond_pad_edge_width_microns=1500,
    template_path="/Users/dtemples/GDS-Geometry-Evaluator/assets/feedlines/feedline-launch-caltech-style.gds",
    layer=3,
    datatype=0,
)

fl_launch_built = gds.build_feedline_launcher(fl_launch_spec)
fl_launch_built.plot()

In [ ]:
fl_spec = gds.StraightFeedlineSpec(
    launcher_spec=fl_launch_spec,
    chip_width_microns=chip_w_um,
    chip_height_microns=chip_h_um,
    face="left",
    offset_microns=0,
    ground_pour_spec=gds.GroundPourSpec(
            chip_edge_border_microns=25,
            structure_gap_microns=5,
        ),
)

built_fl = gds.build_straight_feedline(fl_spec)
built_fl.plot()

In [ ]:
horiz_position_micron = [
    -6500.0, -3250.0, 0.0, 3250.0, 6500.0,
]

In [ ]:
placed = gds.place_lekid_on_feedline(
        feedline_result=built_fl,
        lekid_result=built_lekid_list[0],
        spec=gds.LekidFeedlinePlacementSpec(
            separation_microns=cpw_gap_width+cpw_gnd_width+gs_gap_width,
            position_microns=horiz_position_micron[0],
        )
    )

for i_kid in range(1,2):#N_resonators):
    placed = gds.place_lekid_on_feedline(
        placed,
        lekid_result=built_lekid_list[i_kid],
        spec=gds.LekidFeedlinePlacementSpec(
            separation_microns=cpw_gap_width+cpw_gnd_width+gs_gap_width,
            position_microns=horiz_position_micron[i_kid],
        ),
    )
    
placed.plot()

gds.write_built_gds("./MLA5G-TestMask-1.gds", placed)